# Merge Sentence Datasets (examples + sentences + localization folder)

This notebook merges multiple dataset directories into one combined output directory.

It writes:
- `examples.jsonl`
- `sentences.jsonl`
- `localization/` (per-example JSON files)
- `localization.jsonl` (rebuilt only from files in `localization/`)

Note: this notebook intentionally does **not** read existing `localization*.jsonl` inputs.

In [1]:
from pathlib import Path
from collections import Counter
import glob
import json
import random
import re


def resolve_input_paths(sources, jsonl_name):
    paths = []
    for src in sources:
        if any(ch in src for ch in ['*', '?', '[']):
            paths.extend(Path(p) for p in sorted(glob.glob(src)))
            continue

        p = Path(src)
        if p.is_dir():
            paths.append(p / jsonl_name)
        else:
            paths.append(p)

    return paths


def canonical_dir(path):
    return str(Path(path).resolve())


def make_source_key(path):
    name = Path(path).name or 'source'
    key = re.sub(r'[^A-Za-z0-9._-]+', '_', name).strip('_')
    return key or 'source'


def build_source_key_map(dataset_dirs):
    key_counts = Counter()
    mapping = {}

    for src in dataset_dirs:
        src_dir = canonical_dir(src)
        base_key = make_source_key(src_dir)
        key_counts[base_key] += 1
        idx = key_counts[base_key]
        mapping[src_dir] = base_key if idx == 1 else f"{base_key}_{idx}"

    return mapping


def build_input_suffix_map(dataset_dirs, suffixes=None):
    if suffixes is None:
        suffixes = [''] * len(dataset_dirs)

    if len(suffixes) != len(dataset_dirs):
        raise ValueError(
            f'ID_SUFFIXES_BY_INPUT must have length {len(dataset_dirs)}; '
            f'got {len(suffixes)}'
        )

    mapping = {}
    for src, suffix in zip(dataset_dirs, suffixes):
        mapping[canonical_dir(src)] = '' if suffix is None else str(suffix)

    return mapping


def source_key_for_dir(src_dir, source_key_map):
    src_dir = canonical_dir(src_dir)
    return source_key_map.get(src_dir, make_source_key(src_dir))


def suffix_for_dir(src_dir, suffix_map):
    src_dir = canonical_dir(src_dir)
    return suffix_map.get(src_dir, '')


def namespace_id(source_key, raw_id):
    return f"{source_key}::{raw_id}"


def transform_id(raw_id, *, source_key, id_suffix='', use_namespace=False):
    raw = str(raw_id)
    if use_namespace:
        return namespace_id(source_key, raw)
    if id_suffix:
        return f"{raw}{id_suffix}"
    return raw


def load_jsonl(path):
    rows = []
    with Path(path).open('r', encoding='utf-8') as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f'JSON parse error in {path} line {line_no}: {e}') from e
    return rows


def write_jsonl(rows, out_path):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open('w', encoding='utf-8') as f:
        for rec in rows:
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')


def dedupe_rows(rows, key, keep='first'):
    if key is None:
        return rows
    if keep not in {'first', 'last'}:
        raise ValueError("keep must be 'first' or 'last'")

    kept = {}
    order = []
    missing_idx = 0

    for rec in rows:
        k = rec.get(key)
        if k is None:
            missing_idx += 1
            k = f'__missing__{missing_idx}'

        if k not in kept:
            kept[k] = rec
            order.append(k)
        elif keep == 'last':
            kept[k] = rec

    return [kept[k] for k in order]


def extract_example_id(rec):
    ex_id = rec.get('example_id')
    if ex_id:
        return str(ex_id)

    rec_id = rec.get('record_id')
    if rec_id:
        return str(rec_id)

    run_id = str(rec.get('run_id', 'run'))
    state_id = rec.get('state_id')
    sample_idx = rec.get('sample_idx')
    if state_id is not None and sample_idx is not None:
        return f"{run_id}/state_{state_id}/sample_{sample_idx}"

    game_id = rec.get('game_id')
    turn_idx = rec.get('turn_idx')
    if game_id is not None and turn_idx is not None:
        return f"{run_id}/game_{game_id}/turn_{turn_idx}"

    return None


def safe_example_id(example_id):
    s = str(example_id)
    s = re.sub(r'[^A-Za-z0-9._-]+', '_', s)
    return s.strip('_') or 'example'

In [2]:
# --- Configure here ---
INPUT_DATASET_DIRS = [
    '/playpen-ssd/smerrill/deception2/BS/Results/SentencePipeline/v1/DeepSeek-R1-Distill-Qwen-7B_truthful_3000',
    '/playpen-ssd/smerrill/deception2/BS/Results/SentencePipeline/v1/DeepSeek-R1-Distill-Qwen-7B',
]

EXAMPLES_FILENAME = 'examples.jsonl'
SENTENCES_FILENAME = 'sentences.jsonl'
LOCALIZATION_DIRNAME = 'localization'
LOCALIZATION_FILE_GLOB = 'sentence_localization_*.json'

OUTPUT_DIR = '/playpen-ssd/smerrill/deception2/Dataset/BS/DeepSeek-R1-Distill-Qwen-7B'
OUTPUT_EXAMPLES = str(Path(OUTPUT_DIR) / EXAMPLES_FILENAME)
OUTPUT_SENTENCES = str(Path(OUTPUT_DIR) / SENTENCES_FILENAME)
OUTPUT_LOCALIZATION_DIR = str(Path(OUTPUT_DIR) / LOCALIZATION_DIRNAME)
OUTPUT_LOCALIZATION_JSONL = str(Path(OUTPUT_DIR) / 'localization.jsonl')

ADD_SOURCE_FIELD = True
SOURCE_FIELD = '_merge_source'

# Choose one ID collision strategy:
# 1) NAMESPACE_IDS_BY_SOURCE=True: ids become "<source>::<id>"
# 2) APPEND_ID_SUFFIX_BY_SOURCE=True: ids become "<id><suffix>" per input dataset
NAMESPACE_IDS_BY_SOURCE = False
APPEND_ID_SUFFIX_BY_SOURCE = False
ID_SUFFIXES_BY_INPUT = ['_truthful', '']  # aligns with INPUT_DATASET_DIRS

ORIGINAL_EXAMPLE_ID_FIELD = '_orig_example_id'
ORIGINAL_SENTENCE_ID_FIELD = '_orig_sentence_id'

DEDUPE_EXAMPLES_KEY = 'example_id'     # set to None to disable
DEDUPE_SENTENCES_KEY = 'sentence_id'   # set to None to disable
DEDUPE_LOCALIZATION_KEY = 'example_id' # set to None to disable
DEDUPE_KEEP = 'first'  # 'first' or 'last'

FILTER_SENTENCES_TO_MERGED_EXAMPLES = True
FILTER_LOCALIZATION_TO_MERGED_EXAMPLES = False

SHUFFLE_EXAMPLES = False
SHUFFLE_SEED = 42

LOCALIZATION_FILE_KEEP = 'first'  # 'first' or 'last'

if NAMESPACE_IDS_BY_SOURCE and APPEND_ID_SUFFIX_BY_SOURCE:
    raise ValueError('Set only one of NAMESPACE_IDS_BY_SOURCE or APPEND_ID_SUFFIX_BY_SOURCE to True.')

SOURCE_KEY_BY_DIR = build_source_key_map(INPUT_DATASET_DIRS)
SUFFIX_BY_DIR = build_input_suffix_map(
    INPUT_DATASET_DIRS,
    ID_SUFFIXES_BY_INPUT if APPEND_ID_SUFFIX_BY_SOURCE else None,
)

print('Source key map:')
for src_dir, source_key in SOURCE_KEY_BY_DIR.items():
    print(f'  {source_key}: {src_dir}')

if APPEND_ID_SUFFIX_BY_SOURCE:
    print('ID suffix map:')
    for src_dir, suffix in SUFFIX_BY_DIR.items():
        print(f'  {suffix!r}: {src_dir}')

Source key map:
  DeepSeek-R1-Distill-Qwen-7B_truthful_3000: /playpen-ssd/smerrill/deception2/BS/Results/SentencePipeline/v1/DeepSeek-R1-Distill-Qwen-7B_truthful_3000
  DeepSeek-R1-Distill-Qwen-7B: /playpen-ssd/smerrill/deception2/BS/Results/SentencePipeline/v1/DeepSeek-R1-Distill-Qwen-7B


In [3]:
# Merge examples.jsonl
example_paths = resolve_input_paths(INPUT_DATASET_DIRS, EXAMPLES_FILENAME)
if not example_paths:
    raise ValueError('No example paths found.')

merged_examples = []
example_counts_by_source = Counter()

for p in example_paths:
    if not p.exists():
        raise FileNotFoundError(f'Missing examples file: {p}')

    rows = load_jsonl(p)
    src_dir = canonical_dir(p.parent)
    source_key = source_key_for_dir(src_dir, SOURCE_KEY_BY_DIR)
    id_suffix = suffix_for_dir(src_dir, SUFFIX_BY_DIR)
    example_counts_by_source[source_key] += len(rows)

    for r in rows:
        ex_id = extract_example_id(r)
        if ex_id and not r.get('example_id'):
            r['example_id'] = ex_id

        if r.get('example_id') is not None:
            raw_ex_id = str(r.get('example_id'))
            new_ex_id = transform_id(
                raw_ex_id,
                source_key=source_key,
                id_suffix=id_suffix if APPEND_ID_SUFFIX_BY_SOURCE else '',
                use_namespace=NAMESPACE_IDS_BY_SOURCE,
            )
            if new_ex_id != raw_ex_id:
                r[ORIGINAL_EXAMPLE_ID_FIELD] = raw_ex_id
            r['example_id'] = new_ex_id

        if ADD_SOURCE_FIELD:
            r[SOURCE_FIELD] = source_key

    merged_examples.extend(rows)

before_ex_dedupe = len(merged_examples)
merged_examples = dedupe_rows(merged_examples, DEDUPE_EXAMPLES_KEY, keep=DEDUPE_KEEP)
after_ex_dedupe = len(merged_examples)

if SHUFFLE_EXAMPLES:
    rng = random.Random(SHUFFLE_SEED)
    rng.shuffle(merged_examples)

write_jsonl(merged_examples, OUTPUT_EXAMPLES)

merged_example_ids = {str(r.get('example_id')) for r in merged_examples if r.get('example_id')}

print('Examples merged ->', OUTPUT_EXAMPLES)
print('Rows before dedupe:', before_ex_dedupe)
print('Rows after dedupe :', after_ex_dedupe)
print('Unique example_id  :', len(merged_example_ids))

print('Rows loaded per source:')
for source_key, n in example_counts_by_source.items():
    print(f'  {source_key}: {n}')

Examples merged -> /playpen-ssd/smerrill/deception2/Dataset/BS/DeepSeek-R1-Distill-Qwen-7B/examples.jsonl
Rows before dedupe: 6000
Rows after dedupe : 6000
Unique example_id  : 6000
Rows loaded per source:
  DeepSeek-R1-Distill-Qwen-7B_truthful_3000: 3000
  DeepSeek-R1-Distill-Qwen-7B: 3000


In [4]:
# Merge sentences.jsonl
sentence_paths = resolve_input_paths(INPUT_DATASET_DIRS, SENTENCES_FILENAME)
if not sentence_paths:
    raise ValueError('No sentence paths found.')

merged_sentences = []
sentence_counts_by_source = Counter()

for p in sentence_paths:
    if not p.exists():
        raise FileNotFoundError(f'Missing sentences file: {p}')

    rows = load_jsonl(p)
    src_dir = canonical_dir(p.parent)
    source_key = source_key_for_dir(src_dir, SOURCE_KEY_BY_DIR)
    id_suffix = suffix_for_dir(src_dir, SUFFIX_BY_DIR)
    sentence_counts_by_source[source_key] += len(rows)

    for r in rows:
        ex_id = r.get('example_id')
        if ex_id is not None:
            raw_ex_id = str(ex_id)
            new_ex_id = transform_id(
                raw_ex_id,
                source_key=source_key,
                id_suffix=id_suffix if APPEND_ID_SUFFIX_BY_SOURCE else '',
                use_namespace=NAMESPACE_IDS_BY_SOURCE,
            )
            if new_ex_id != raw_ex_id:
                r[ORIGINAL_EXAMPLE_ID_FIELD] = raw_ex_id
            r['example_id'] = new_ex_id

        sent_id = r.get('sentence_id')
        if sent_id is not None:
            raw_sent_id = str(sent_id)
            new_sent_id = transform_id(
                raw_sent_id,
                source_key=source_key,
                id_suffix=id_suffix if APPEND_ID_SUFFIX_BY_SOURCE else '',
                use_namespace=NAMESPACE_IDS_BY_SOURCE,
            )
            if new_sent_id != raw_sent_id:
                r[ORIGINAL_SENTENCE_ID_FIELD] = raw_sent_id
            r['sentence_id'] = new_sent_id

        if ADD_SOURCE_FIELD:
            r[SOURCE_FIELD] = source_key

    merged_sentences.extend(rows)

before_sent_filter = len(merged_sentences)
if FILTER_SENTENCES_TO_MERGED_EXAMPLES:
    merged_sentences = [
        r for r in merged_sentences
        if r.get('example_id') is not None and str(r.get('example_id')) in merged_example_ids
    ]
after_sent_filter = len(merged_sentences)

before_sent_dedupe = len(merged_sentences)
merged_sentences = dedupe_rows(merged_sentences, DEDUPE_SENTENCES_KEY, keep=DEDUPE_KEEP)
after_sent_dedupe = len(merged_sentences)

write_jsonl(merged_sentences, OUTPUT_SENTENCES)

print('Sentences merged ->', OUTPUT_SENTENCES)
print('Rows before filter :', before_sent_filter)
print('Rows after filter  :', after_sent_filter)
print('Rows before dedupe :', before_sent_dedupe)
print('Rows after dedupe  :', after_sent_dedupe)

print('Rows loaded per source:')
for source_key, n in sentence_counts_by_source.items():
    print(f'  {source_key}: {n}')

Sentences merged -> /playpen-ssd/smerrill/deception2/Dataset/BS/DeepSeek-R1-Distill-Qwen-7B/sentences.jsonl
Rows before filter : 239270
Rows after filter  : 239270
Rows before dedupe : 239270
Rows after dedupe  : 239270
Rows loaded per source:
  DeepSeek-R1-Distill-Qwen-7B_truthful_3000: 111117
  DeepSeek-R1-Distill-Qwen-7B: 128153


In [ ]:
# Build output localization/ and localization.jsonl from localization/*.json files only
output_loc_dir = Path(OUTPUT_LOCALIZATION_DIR)
output_loc_dir.mkdir(parents=True, exist_ok=True)

input_loc_files = []
for src_dir in INPUT_DATASET_DIRS:
    loc_dir = Path(src_dir) / LOCALIZATION_DIRNAME
    if loc_dir.exists():
        input_loc_files.extend(sorted(loc_dir.glob(LOCALIZATION_FILE_GLOB)))

kept = {}
order = []
loaded_count = 0
skipped_missing_example_id = 0
skipped_not_in_examples = 0

for p in input_loc_files:
    try:
        rec = json.loads(p.read_text(encoding='utf-8'))
    except Exception:
        continue

    loaded_count += 1
    ex_id = rec.get('example_id')
    if not ex_id:
        skipped_missing_example_id += 1
        continue

    src_dir = canonical_dir(p.parent.parent)
    source_key = source_key_for_dir(src_dir, SOURCE_KEY_BY_DIR)
    id_suffix = suffix_for_dir(src_dir, SUFFIX_BY_DIR)

    raw_ex_id = str(ex_id)
    new_ex_id = transform_id(
        raw_ex_id,
        source_key=source_key,
        id_suffix=id_suffix if APPEND_ID_SUFFIX_BY_SOURCE else '',
        use_namespace=NAMESPACE_IDS_BY_SOURCE,
    )
    if new_ex_id != raw_ex_id:
        rec[ORIGINAL_EXAMPLE_ID_FIELD] = raw_ex_id
    rec['example_id'] = new_ex_id

    if FILTER_LOCALIZATION_TO_MERGED_EXAMPLES and new_ex_id not in merged_example_ids:
        skipped_not_in_examples += 1
        continue

    if ADD_SOURCE_FIELD:
        rec[SOURCE_FIELD] = source_key

    dedupe_key = rec.get(DEDUPE_LOCALIZATION_KEY) if DEDUPE_LOCALIZATION_KEY else f"__idx__{loaded_count}"
    if dedupe_key not in kept:
        kept[dedupe_key] = rec
        order.append(dedupe_key)
    elif LOCALIZATION_FILE_KEEP == 'last':
        kept[dedupe_key] = rec

localization_rows = [kept[k] for k in order]

written_files = 0
for rec in localization_rows:
    ex_id = str(rec.get('example_id'))
    out_name = f"sentence_localization_{safe_example_id(ex_id)}.json"
    out_path = output_loc_dir / out_name
    out_path.write_text(json.dumps(rec, indent=2, ensure_ascii=False), encoding='utf-8')
    written_files += 1

write_jsonl(localization_rows, OUTPUT_LOCALIZATION_JSONL)

print('Localization files input :', len(input_loc_files))
print('Localization JSON parsed :', loaded_count)
print('Skipped missing example_id:', skipped_missing_example_id)
print('Skipped (not in examples):', skipped_not_in_examples)
print('Localization files written:', written_files, '->', OUTPUT_LOCALIZATION_DIR)
print('Localization JSONL written:', len(localization_rows), '->', OUTPUT_LOCALIZATION_JSONL)

In [ ]:
# Final checks
final_examples = load_jsonl(OUTPUT_EXAMPLES)
final_sentences = load_jsonl(OUTPUT_SENTENCES)
final_localization = load_jsonl(OUTPUT_LOCALIZATION_JSONL)

example_ids = {str(r.get('example_id')) for r in final_examples if r.get('example_id')}
sentence_ids = {str(r.get('example_id')) for r in final_sentences if r.get('example_id')}
loc_ids = {str(r.get('example_id')) for r in final_localization if r.get('example_id')}

print('examples rows      :', len(final_examples))
print('sentences rows     :', len(final_sentences))
print('localization rows  :', len(final_localization))
print('example_id in examples    :', len(example_ids))
print('example_id in sentences   :', len(sentence_ids))
print('example_id in localization:', len(loc_ids))
print('sentences not in examples :', len(sentence_ids - example_ids))
print('localization not in examples:', len(loc_ids - example_ids))

loc_file_count = len(list(Path(OUTPUT_LOCALIZATION_DIR).glob('sentence_localization_*.json')))
print('localization/ file count:', loc_file_count)

print('\nOutput dir:', OUTPUT_DIR)
print('  -', OUTPUT_EXAMPLES)
print('  -', OUTPUT_SENTENCES)
print('  -', OUTPUT_LOCALIZATION_JSONL)
print('  -', OUTPUT_LOCALIZATION_DIR)

examples rows      : 6000
sentences rows     : 222234
localization rows  : 6000
example_id in examples    : 6000
example_id in sentences   : 6000
example_id in localization: 6000
sentences not in examples : 0
localization not in examples: 3000
localization/ file count: 6000

Output dir: /playpen-ssd/smerrill/deception2/Dataset/BS/DeepSeek-R1-Distill-Qwen-7B_combined
  - /playpen-ssd/smerrill/deception2/Dataset/BS/DeepSeek-R1-Distill-Qwen-7B_combined/examples.jsonl
  - /playpen-ssd/smerrill/deception2/Dataset/BS/DeepSeek-R1-Distill-Qwen-7B_combined/sentences.jsonl
  - /playpen-ssd/smerrill/deception2/Dataset/BS/DeepSeek-R1-Distill-Qwen-7B_combined/localization.jsonl
  - /playpen-ssd/smerrill/deception2/Dataset/BS/DeepSeek-R1-Distill-Qwen-7B_combined/localization


### Test linking

In [ ]:
# Link validation cell (mirrors dashboard/app.py join behavior)
from pathlib import Path
from collections import Counter
import gzip
import json
import re

def normalize_example_id(example_id):
    if not isinstance(example_id, str):
        return ""
    return example_id.strip()

def _safe_int(value):
    try:
        if value is None:
            return None
        return int(value)
    except (TypeError, ValueError):
        return None

def resolve_jsonl_or_gz(path: Path):
    if path.exists():
        return path
    gz_path = path.with_name(path.name + ".gz")
    if gz_path.exists():
        return gz_path
    return path

def open_text_maybe_gzip(path: Path):
    if path.suffix == ".gz":
        return gzip.open(path, "rt", encoding="utf-8")
    return path.open("r", encoding="utf-8")

def load_jsonl(path: Path):
    path = resolve_jsonl_or_gz(path)
    out = []
    with open_text_maybe_gzip(path) as f:
        for line in f:
            line = line.strip()
            if line:
                out.append(json.loads(line))
    return out

def load_sentences_index(path: Path, include_example_ids=None):
    path = resolve_jsonl_or_gz(path)
    include_set = set(include_example_ids) if include_example_ids else None
    index = {}
    with open_text_maybe_gzip(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            ex_id = normalize_example_id(rec.get("example_id"))
            if not ex_id:
                continue
            if include_set is not None and ex_id not in include_set:
                continue

            sent_idx = _safe_int(rec.get("sentence_idx"))
            if sent_idx is None:
                continue

            index.setdefault(ex_id, []).append({
                "sentence_idx": sent_idx,
                "sentence_id": rec.get("sentence_id"),
                "sentence_text": rec.get("sentence_text", ""),
                "start": _safe_int(rec.get("start")),
                "end": _safe_int(rec.get("end")),
            })

    for ex_id in list(index.keys()):
        index[ex_id] = sorted(index[ex_id], key=lambda r: r.get("sentence_idx", 0))
    return index

from pathlib import Path

examples_path = Path("/playpen-ssd/smerrill/deception2/Dataset/BS/DeepSeek-R1-Distill-Qwen-7B_combined/examples.jsonl")
sentences_path = Path("/playpen-ssd/smerrill/deception2/Dataset/BS/DeepSeek-R1-Distill-Qwen-7B_combined/sentences.jsonl")
localization_path = Path("/playpen-ssd/smerrill/deception2/Dataset/BS/DeepSeek-R1-Distill-Qwen-7B_combined/localization.jsonl")

print("examples_path:", examples_path)
print("sentences_path:", sentences_path)
print("localization_path:", localization_path)

examples = load_jsonl(examples_path)
localization = load_jsonl(localization_path)

loc_example_ids = sorted({
    normalize_example_id(r.get("example_id"))
    for r in localization
    if normalize_example_id(r.get("example_id"))
})

examples_index = {
    normalize_example_id(r.get("example_id")): r
    for r in examples
    if normalize_example_id(r.get("example_id"))
}

sentences_index = load_sentences_index(sentences_path, include_example_ids=tuple(loc_example_ids))

missing_in_examples = [ex for ex in loc_example_ids if ex not in examples_index]
missing_in_sentences = [ex for ex in loc_example_ids if ex not in sentences_index]

# Span/text integrity check against localization raw_text
invalid_span_rows = 0
text_mismatch_rows = 0
checked_span_rows = 0
text_compare_rows = 0
mismatch_samples = []

for rec in localization:
    ex_id = normalize_example_id(rec.get("example_id"))
    raw_text = rec.get("raw_text") if isinstance(rec.get("raw_text"), str) else ""
    for s in sentences_index.get(ex_id, []):
        start = s.get("start")
        end = s.get("end")
        if start is None or end is None or end <= start or (raw_text and end > len(raw_text)):
            invalid_span_rows += 1
            if len(mismatch_samples) < 10:
                mismatch_samples.append({
                    "example_id": ex_id,
                    "reason": "invalid_span",
                    "start": start,
                    "end": end,
                    "raw_len": len(raw_text),
                    "sentence_idx": s.get("sentence_idx"),
                })
            continue

        checked_span_rows += 1
        sent_text = s.get("sentence_text") or ""
        if raw_text and sent_text:
            text_compare_rows += 1
            span_text = raw_text[start:end]
            if span_text.strip() != sent_text.strip():
                text_mismatch_rows += 1
                if len(mismatch_samples) < 10:
                    mismatch_samples.append({
                        "example_id": ex_id,
                        "reason": "span_text_mismatch",
                        "sentence_idx": s.get("sentence_idx"),
                        "span_text": span_text[:120],
                        "sentence_text": sent_text[:120],
                    })

# Optional: check example text field alignment with localization raw_text
example_text_field_hits = Counter()
example_text_checked = 0
for rec in localization:
    ex_id = normalize_example_id(rec.get("example_id"))
    raw_text = rec.get("raw_text") if isinstance(rec.get("raw_text"), str) else ""
    ex = examples_index.get(ex_id)
    if not ex or not raw_text:
        continue
    example_text_checked += 1
    for fld in ("action_reasoning", "action_raw_text"):
        v = ex.get(fld)
        if isinstance(v, str) and v.strip() == raw_text.strip():
            example_text_field_hits[fld] += 1

print("=== Link Coverage (Visualizer-style) ===")
print(f"Localization rows: {len(localization)}")
print(f"Localization unique example_id: {len(loc_example_ids)}")
print(f"Examples unique example_id: {len(examples_index)}")
print(f"Sentences example_id keys: {len(sentences_index)}")
print(f"Missing in examples: {len(missing_in_examples)}")
print(f"Missing in sentences: {len(missing_in_sentences)}")

print("\n=== Span Integrity ===")
print(f"Checked sentence spans: {checked_span_rows}")
print(f"Invalid span rows: {invalid_span_rows}")
print(f"Text-compared spans: {text_compare_rows}")
print(f"Span text mismatches: {text_mismatch_rows}")

print("\n=== Example Text Alignment ===")
print(f"Localization rows with matching example present + raw_text: {example_text_checked}")
print(f"Exact matches by field: {dict(example_text_field_hits)}")

if missing_in_examples:
    print("\nSample missing_in_examples:", missing_in_examples[:10])
if missing_in_sentences:
    print("\nSample missing_in_sentences:", missing_in_sentences[:10])
if mismatch_samples:
    print("\nSample mismatch rows:")
    for row in mismatch_samples[:10]:
        print(row)


examples_path: /playpen-ssd/smerrill/deception2/Dataset/BS/DeepSeek-R1-Distill-Qwen-7B_combined/examples.jsonl
sentences_path: /playpen-ssd/smerrill/deception2/Dataset/BS/DeepSeek-R1-Distill-Qwen-7B_combined/sentences.jsonl
localization_path: /playpen-ssd/smerrill/deception2/Dataset/BS/DeepSeek-R1-Distill-Qwen-7B_combined/localization.jsonl
=== Link Coverage (Visualizer-style) ===
Localization rows: 6000
Localization unique example_id: 6000
Examples unique example_id: 6000
Sentences example_id keys: 3000
Missing in examples: 3000
Missing in sentences: 3000

=== Span Integrity ===
Checked sentence spans: 111117
Invalid span rows: 0
Text-compared spans: 111117
Span text mismatches: 0

=== Example Text Alignment ===
Localization rows with matching example present + raw_text: 3000
Exact matches by field: {'action_reasoning': 3000}

Sample missing_in_examples: ['2026-02-06/gpu_2/state_100/sample_0', '2026-02-06/gpu_2/state_1002/sample_0', '2026-02-06/gpu_2/state_1005/sample_0', '2026-02-06/